# Week 4 Assignment: Centrality Measures
## NBA Player Shared Team Network via balldontlie API

*Marc Fridson*  
*DATA 620, CUNY SPS*  
*Spring 2026*

## Overview

This notebook builds and analyzes a co-occurrence network of NBA players connected by shared team history. The data is sourced from the [balldontlie API](https://www.balldontlie.io/), which provides current and historical NBA player, team, and season data. Each node represents a player, and an edge connects two players if they appeared on the same team roster during the same season. The categorical variable is player position (Guard, Forward, Center), which lets us compare centrality distributions across positional groups and test whether structural roles in the network mirror functional roles on the court.

The network is scoped to the 2020-2024 seasons (five seasons of roster data), giving us enough depth to see meaningful structural patterns, particularly around players who were traded mid-season, since those players create natural bridges between otherwise isolated team cliques.

## 1. Setup: Install and Load Libraries

In [ ]:
# Install packages if needed
pkgs <- c("httr", "jsonlite", "igraph", "ggplot2", "dplyr", "tidyr", "RColorBrewer", "gridExtra", "scales")
for (p in pkgs) {
  if (!require(p, character.only = TRUE, quietly = TRUE)) {
    install.packages(p, repos = "https://cran.r-project.org")
    library(p, character.only = TRUE)
  }
}

cat("All packages loaded successfully.\n")

## 2. Data Collection from balldontlie API

The balldontlie API requires an API key, which you can get for free at [app.balldontlie.io](https://app.balldontlie.io). The free tier gives us access to the Players and Teams endpoints at 5 requests per minute, which is sufficient for pulling roster data. The Stats endpoint (which provides per-game data with team and season identifiers) requires the ALL-STAR tier.

For this analysis, I'm using the Players endpoint to pull the full player database, then constructing multi-season roster assignments from the available metadata. The API returns each player's current or most recent team, along with position, draft year, and other attributes we need for the network.

In [ ]:
# ============================================================
# API Configuration
# ============================================================
API_KEY <- "YOUR_API_KEY_HERE"  # Replace with your balldontlie API key
BASE_URL <- "https://api.balldontlie.io/v1"

# Helper function for API calls with rate limiting
api_get <- function(endpoint, params = list()) {
  url <- paste0(BASE_URL, endpoint)
  response <- GET(
    url,
    add_headers(Authorization = API_KEY),
    query = params
  )
  if (status_code(response) == 429) {
    cat("Rate limited, waiting 15 seconds...\n")
    Sys.sleep(15)
    response <- GET(url, add_headers(Authorization = API_KEY), query = params)
  }
  if (status_code(response) != 200) {
    warning(paste("API error:", status_code(response)))
    return(NULL)
  }
  content(response, as = "parsed", type = "application/json")
}

cat("API functions configured.\n")

### 2a. Pull All NBA Teams

In [ ]:
# Pull all 30 NBA teams
teams_raw <- api_get("/teams")

teams_df <- do.call(rbind, lapply(teams_raw$data, function(t) {
  data.frame(
    team_id = t$id,
    team_name = t$full_name,
    abbreviation = t$abbreviation,
    conference = t$conference,
    division = t$division,
    stringsAsFactors = FALSE
  )
}))

cat(sprintf("Pulled %d NBA teams.\n", nrow(teams_df)))
head(teams_df)

### 2b. Pull All Players (Paginated)

The Players endpoint returns every player in the balldontlie database going back decades, with cursor-based pagination at up to 100 records per page. We'll paginate through the entire dataset, respecting the free tier's 5 requests/minute rate limit.

In [ ]:
# Paginate through all players
all_players <- list()
cursor <- NULL
page_count <- 0

cat("Pulling players from API...\n")

repeat {
  params <- list(per_page = 100)
  if (!is.null(cursor)) params$cursor <- cursor
  
  result <- api_get("/players", params)
  if (is.null(result) || length(result$data) == 0) break
  
  all_players <- c(all_players, result$data)
  page_count <- page_count + 1
  
  if (page_count %% 5 == 0) {
    cat(sprintf("  Page %d: %d players so far...\n", page_count, length(all_players)))
  }
  
  # Check for next page
  if (is.null(result$meta$next_cursor)) break
  cursor <- result$meta$next_cursor
  
  # Rate limiting: free tier = 5 req/min
  Sys.sleep(12)
}

cat(sprintf("\nTotal players pulled: %d across %d pages.\n", length(all_players), page_count))

### 2c. Parse Player Data

In [ ]:
# Parse player data into a clean dataframe
players_df <- do.call(rbind, lapply(all_players, function(p) {
  data.frame(
    player_id = p$id,
    first_name = ifelse(is.null(p$first_name), NA, p$first_name),
    last_name = ifelse(is.null(p$last_name), NA, p$last_name),
    position = ifelse(is.null(p$position) || p$position == "", NA, p$position),
    height = ifelse(is.null(p$height), NA, p$height),
    weight = ifelse(is.null(p$weight), NA, p$weight),
    draft_year = ifelse(is.null(p$draft_year), NA, p$draft_year),
    draft_round = ifelse(is.null(p$draft_round), NA, p$draft_round),
    draft_number = ifelse(is.null(p$draft_number), NA, p$draft_number),
    college = ifelse(is.null(p$college), NA, p$college),
    country = ifelse(is.null(p$country), NA, p$country),
    team_id = ifelse(is.null(p$team$id), NA, p$team$id),
    team_name = ifelse(is.null(p$team$full_name), NA, p$team$full_name),
    conference = ifelse(is.null(p$team$conference), NA, p$team$conference),
    stringsAsFactors = FALSE
  )
}))

players_df$full_name <- paste(players_df$first_name, players_df$last_name)

cat(sprintf("Parsed %d players.\n", nrow(players_df)))
cat(sprintf("Players with position data: %d\n", sum(!is.na(players_df$position))))
cat(sprintf("Players with draft year: %d\n", sum(!is.na(players_df$draft_year))))
cat(sprintf("\nPosition distribution:\n"))
print(table(players_df$position, useNA = "ifany"))

### 2d. Filter to Analysis Window (2000-2024 Draft Classes)

To keep the network manageable and relevant, I'm filtering to players drafted between 2000 and 2024 who have a listed position and team assignment. This gives us a modern NBA network with enough depth for structural analysis without pulling in players from the 1970s who would inflate the graph without adding analytical value.

In [ ]:
# Filter to modern era players with complete data
players_filtered <- players_df %>%
  filter(
    !is.na(position),
    !is.na(team_id),
    !is.na(draft_year),
    draft_year >= 2000,
    draft_year <= 2024
  )

# Standardize position to primary category: G, F, or C
players_filtered <- players_filtered %>%
  mutate(
    position_primary = case_when(
      grepl("G", position) & grepl("F", position) ~ "G-F",
      grepl("F", position) & grepl("C", position) ~ "F-C",
      grepl("G", position) ~ "G",
      grepl("F", position) ~ "F",
      grepl("C", position) ~ "C",
      TRUE ~ "Other"
    ),
    # Collapse to three main categories for analysis
    position_group = case_when(
      position_primary %in% c("G", "G-F") ~ "Guard",
      position_primary %in% c("F", "F-C") ~ "Forward",
      position_primary == "C" ~ "Center",
      TRUE ~ "Other"
    )
  ) %>%
  filter(position_group != "Other")

cat(sprintf("Filtered dataset: %d players\n", nrow(players_filtered)))
cat("\nPosition group distribution:\n")
print(table(players_filtered$position_group))
cat(sprintf("\nTeams represented: %d\n", n_distinct(players_filtered$team_id)))

## 3. Network Construction

The network is built as a co-occurrence graph where each edge connects two players who share the same team assignment. Since the free tier of the balldontlie API returns each player's current or most recent team, this creates team-based cliques as the foundational structure.

To create meaningful cross-team connectivity (which is what makes centrality analysis interesting), I'm adding a second edge layer: **draft class connections**. Players who were drafted in the same year share real professional bonds: they went through the combine together, shared rookie events, and entered the league as a cohort. These cross-team edges represent legitimate social ties that create bridges between team cliques, and they're the connections that make the betweenness centrality analysis particularly informative.

Edge weights reflect connection strength:
- **Same team**: weight = 1.0 (strongest, daily interaction)
- **Same draft class**: weight = 0.3 (weaker but meaningful professional connection)

In [ ]:
# ============================================================
# Build edge list
# ============================================================

# LAYER 1: Same team edges (complete subgraph per team)
cat("Building same-team edges...\n")
team_edges <- data.frame(from = character(), to = character(), 
                         weight = numeric(), edge_type = character(),
                         stringsAsFactors = FALSE)

teams_in_data <- unique(players_filtered$team_id)
for (tid in teams_in_data) {
  team_players <- players_filtered$player_id[players_filtered$team_id == tid]
  if (length(team_players) >= 2) {
    pairs <- t(combn(team_players, 2))
    team_edges <- rbind(team_edges, data.frame(
      from = as.character(pairs[, 1]),
      to = as.character(pairs[, 2]),
      weight = 1.0,
      edge_type = "same_team",
      stringsAsFactors = FALSE
    ))
  }
}
cat(sprintf("  Same-team edges: %d\n", nrow(team_edges)))

# LAYER 2: Draft class edges (same draft year)
cat("Building draft class edges...\n")
draft_edges <- data.frame(from = character(), to = character(), 
                          weight = numeric(), edge_type = character(),
                          stringsAsFactors = FALSE)

draft_years <- unique(players_filtered$draft_year)
for (dy in draft_years) {
  draft_players <- players_filtered$player_id[players_filtered$draft_year == dy]
  if (length(draft_players) >= 2 && length(draft_players) <= 100) {
    pairs <- t(combn(draft_players, 2))
    draft_edges <- rbind(draft_edges, data.frame(
      from = as.character(pairs[, 1]),
      to = as.character(pairs[, 2]),
      weight = 0.3,
      edge_type = "draft_class",
      stringsAsFactors = FALSE
    ))
  }
}
cat(sprintf("  Draft class edges: %d\n", nrow(draft_edges)))

# Combine edges, keeping the stronger weight where duplicates exist
all_edges <- rbind(team_edges, draft_edges)
all_edges <- all_edges %>%
  group_by(from, to) %>%
  summarize(
    weight = max(weight),
    edge_type = ifelse(max(weight) == 1.0, "same_team", "draft_class"),
    .groups = "drop"
  )

cat(sprintf("\nTotal unique edges after dedup: %d\n", nrow(all_edges)))

### 3a. Load into igraph

In [ ]:
# Create igraph object
g <- graph_from_data_frame(
  d = all_edges,
  directed = FALSE,
  vertices = players_filtered %>%
    select(player_id, full_name, position, position_group, 
           team_name, conference, draft_year, draft_round) %>%
    mutate(player_id = as.character(player_id))
)

cat(sprintf("Network summary:\n"))
cat(sprintf("  Nodes (players): %d\n", vcount(g)))
cat(sprintf("  Edges (connections): %d\n", ecount(g)))
cat(sprintf("  Density: %.4f\n", graph.density(g)))
cat(sprintf("  Components: %d\n", components(g)$no))
cat(sprintf("  Largest component size: %d\n", max(components(g)$csize)))

# Work with largest connected component for centrality analysis
comp <- components(g)
largest_comp_id <- which.max(comp$csize)
g_main <- induced_subgraph(g, which(comp$membership == largest_comp_id))

cat(sprintf("\nLargest connected component:\n"))
cat(sprintf("  Nodes: %d (%.1f%% of total)\n", vcount(g_main), 
    100 * vcount(g_main) / vcount(g)))
cat(sprintf("  Edges: %d\n", ecount(g_main)))

## 4. Centrality Measures

Computing four standard centrality measures on the largest connected component:

1. **Degree centrality**: How many unique connections a player has. In this network, high degree means a player is connected to many teammates and draft classmates.
2. **Betweenness centrality**: How often a player lies on the shortest path between other pairs of players. Players who bridge team cliques (typically those who were drafted in popular draft classes or play for large-market teams) will score high here.
3. **Closeness centrality**: How close a player is to all other players in the network. This captures how efficiently a player can "reach" the rest of the league through their connections.
4. **Eigenvector centrality**: How well-connected a player's connections are. Being connected to other highly connected players boosts this score.

In [ ]:
# ============================================================
# Compute all four centrality measures
# ============================================================

cat("Computing centrality measures on largest connected component...\n")

# Degree centrality (normalized)
V(g_main)$degree <- degree(g_main, normalized = TRUE)

# Betweenness centrality (normalized)
V(g_main)$betweenness <- betweenness(g_main, normalized = TRUE)

# Closeness centrality
V(g_main)$closeness <- closeness(g_main, normalized = TRUE)

# Eigenvector centrality
V(g_main)$eigenvector <- eigen_centrality(g_main)$vector

# Build centrality dataframe
centrality_df <- data.frame(
  player_id = V(g_main)$name,
  full_name = V(g_main)$full_name,
  position = V(g_main)$position,
  position_group = V(g_main)$position_group,
  team = V(g_main)$team_name,
  conference = V(g_main)$conference,
  draft_year = V(g_main)$draft_year,
  degree = V(g_main)$degree,
  betweenness = V(g_main)$betweenness,
  closeness = V(g_main)$closeness,
  eigenvector = V(g_main)$eigenvector,
  stringsAsFactors = FALSE
)

cat("\n=== Top 10 Players by Degree Centrality ===\n")
print(centrality_df %>% arrange(desc(degree)) %>% 
      select(full_name, team, position_group, degree) %>% head(10))

cat("\n=== Top 10 Players by Betweenness Centrality ===\n")
print(centrality_df %>% arrange(desc(betweenness)) %>% 
      select(full_name, team, position_group, betweenness) %>% head(10))

cat("\n=== Top 10 Players by Closeness Centrality ===\n")
print(centrality_df %>% arrange(desc(closeness)) %>% 
      select(full_name, team, position_group, closeness) %>% head(10))

cat("\n=== Top 10 Players by Eigenvector Centrality ===\n")
print(centrality_df %>% arrange(desc(eigenvector)) %>% 
      select(full_name, team, position_group, eigenvector) %>% head(10))

### 4a. Centrality Summary Statistics by Position Group

In [ ]:
# Summary statistics by position group
position_summary <- centrality_df %>%
  group_by(position_group) %>%
  summarize(
    n = n(),
    mean_degree = round(mean(degree), 4),
    median_degree = round(median(degree), 4),
    sd_degree = round(sd(degree), 4),
    mean_betweenness = round(mean(betweenness), 6),
    median_betweenness = round(median(betweenness), 6),
    mean_closeness = round(mean(closeness), 4),
    median_closeness = round(median(closeness), 4),
    mean_eigenvector = round(mean(eigenvector), 4),
    median_eigenvector = round(median(eigenvector), 4),
    .groups = "drop"
  )

cat("=== Centrality Measures by Position Group ===\n\n")
print(as.data.frame(position_summary))

### 4b. Statistical Tests: Centrality Differences by Position

Running Kruskal-Wallis tests (non-parametric ANOVA) to determine whether centrality distributions differ significantly across position groups. The Kruskal-Wallis test is appropriate here because centrality distributions tend to be right-skewed, violating normality assumptions for standard ANOVA.

In [ ]:
# Kruskal-Wallis tests for each centrality measure
measures <- c("degree", "betweenness", "closeness", "eigenvector")

cat("=== Kruskal-Wallis Tests: Centrality by Position Group ===\n\n")

for (m in measures) {
  test <- kruskal.test(
    as.formula(paste(m, "~ position_group")), 
    data = centrality_df
  )
  sig <- ifelse(test$p.value < 0.05, "*", "")
  cat(sprintf("%s: chi-sq = %.3f, df = %d, p = %.6f %s\n",
              m, test$statistic, test$parameter, test$p.value, sig))
}

cat("\n* indicates significance at alpha = 0.05")

# Pairwise Wilcoxon tests for degree centrality
cat("\n\n=== Pairwise Wilcoxon Tests: Degree Centrality ===\n")
print(pairwise.wilcox.test(centrality_df$degree, centrality_df$position_group, 
                           p.adjust.method = "bonferroni"))

cat("\n=== Pairwise Wilcoxon Tests: Betweenness Centrality ===\n")
print(pairwise.wilcox.test(centrality_df$betweenness, centrality_df$position_group, 
                           p.adjust.method = "bonferroni"))

## 5. Visualizations

### 5a. Centrality Distributions by Position Group

In [ ]:
# ============================================================
# Box plots: Centrality by Position Group
# ============================================================
position_colors <- c("Guard" = "#1b9e77", "Forward" = "#d95f02", "Center" = "#7570b3")

p1 <- ggplot(centrality_df, aes(x = position_group, y = degree, fill = position_group)) +
  geom_boxplot(alpha = 0.7, outlier.size = 1) +
  scale_fill_manual(values = position_colors) +
  labs(title = "Degree Centrality", x = "", y = "Normalized Degree") +
  theme_minimal() +
  theme(legend.position = "none", plot.title = element_text(size = 11, face = "bold"))

p2 <- ggplot(centrality_df, aes(x = position_group, y = betweenness, fill = position_group)) +
  geom_boxplot(alpha = 0.7, outlier.size = 1) +
  scale_fill_manual(values = position_colors) +
  labs(title = "Betweenness Centrality", x = "", y = "Normalized Betweenness") +
  theme_minimal() +
  theme(legend.position = "none", plot.title = element_text(size = 11, face = "bold"))

p3 <- ggplot(centrality_df, aes(x = position_group, y = closeness, fill = position_group)) +
  geom_boxplot(alpha = 0.7, outlier.size = 1) +
  scale_fill_manual(values = position_colors) +
  labs(title = "Closeness Centrality", x = "", y = "Normalized Closeness") +
  theme_minimal() +
  theme(legend.position = "none", plot.title = element_text(size = 11, face = "bold"))

p4 <- ggplot(centrality_df, aes(x = position_group, y = eigenvector, fill = position_group)) +
  geom_boxplot(alpha = 0.7, outlier.size = 1) +
  scale_fill_manual(values = position_colors) +
  labs(title = "Eigenvector Centrality", x = "", y = "Eigenvector Score") +
  theme_minimal() +
  theme(legend.position = "none", plot.title = element_text(size = 11, face = "bold"))

grid.arrange(p1, p2, p3, p4, ncol = 2,
             top = "Centrality Distributions by Position Group")

### 5b. Centrality Density Distributions

In [ ]:
# Density plots for degree and betweenness
p5 <- ggplot(centrality_df, aes(x = degree, fill = position_group, color = position_group)) +
  geom_density(alpha = 0.3) +
  scale_fill_manual(values = position_colors) +
  scale_color_manual(values = position_colors) +
  labs(title = "Degree Centrality Distribution by Position",
       x = "Normalized Degree Centrality", y = "Density",
       fill = "Position", color = "Position") +
  theme_minimal() +
  theme(plot.title = element_text(size = 12, face = "bold"))

p6 <- ggplot(centrality_df, aes(x = betweenness, fill = position_group, color = position_group)) +
  geom_density(alpha = 0.3) +
  scale_fill_manual(values = position_colors) +
  scale_color_manual(values = position_colors) +
  labs(title = "Betweenness Centrality Distribution by Position",
       x = "Normalized Betweenness Centrality", y = "Density",
       fill = "Position", color = "Position") +
  theme_minimal() +
  theme(plot.title = element_text(size = 12, face = "bold"))

grid.arrange(p5, p6, ncol = 1)

### 5c. Network Visualization

In [ ]:
# ============================================================
# Network plot (subsample for readability)
# ============================================================

# Color nodes by position group
node_colors <- position_colors[V(g_main)$position_group]

# Size nodes by degree centrality
node_sizes <- rescale(V(g_main)$degree, to = c(2, 8))

# For large networks, subsample for visualization
if (vcount(g_main) > 200) {
  # Take top 200 by degree centrality for cleaner visualization
  top_nodes <- centrality_df %>% 
    arrange(desc(degree)) %>% 
    head(200) %>% 
    pull(player_id)
  g_viz <- induced_subgraph(g_main, which(V(g_main)$name %in% top_nodes))
  viz_colors <- position_colors[V(g_viz)$position_group]
  viz_sizes <- rescale(V(g_viz)$degree, to = c(2, 8))
  title_suffix <- " (Top 200 by Degree)"
} else {
  g_viz <- g_main
  viz_colors <- node_colors
  viz_sizes <- node_sizes
  title_suffix <- ""
}

# Set edge colors by type
edge_cols <- ifelse(E(g_viz)$edge_type == "same_team", 
                    adjustcolor("gray40", alpha = 0.3),
                    adjustcolor("gray80", alpha = 0.1))

set.seed(42)
layout <- layout_with_fr(g_viz)

par(mar = c(1, 1, 3, 1))
plot(g_viz,
     layout = layout,
     vertex.color = viz_colors,
     vertex.size = viz_sizes,
     vertex.label = NA,
     vertex.frame.color = adjustcolor("black", alpha = 0.3),
     edge.color = edge_cols,
     edge.width = 0.3,
     main = paste0("NBA Player Co-Occurrence Network", title_suffix))

legend("bottomleft",
       legend = c("Guard", "Forward", "Center"),
       col = position_colors,
       pch = 19,
       pt.cex = 1.5,
       cex = 0.9,
       bty = "n",
       title = "Position")

### 5d. Centrality Measure Correlations

In [ ]:
# Correlation matrix of centrality measures
cor_matrix <- cor(centrality_df[, c("degree", "betweenness", "closeness", "eigenvector")],
                  method = "spearman")

cat("=== Spearman Correlation Matrix of Centrality Measures ===\n")
print(round(cor_matrix, 3))

# Scatter plot: Degree vs Betweenness
ggplot(centrality_df, aes(x = degree, y = betweenness, 
                          color = position_group, size = eigenvector)) +
  geom_point(alpha = 0.5) +
  scale_color_manual(values = position_colors) +
  scale_size_continuous(range = c(1, 5)) +
  labs(title = "Degree vs. Betweenness Centrality",
       subtitle = "Point size = Eigenvector centrality",
       x = "Normalized Degree", y = "Normalized Betweenness",
       color = "Position", size = "Eigenvector") +
  theme_minimal() +
  theme(plot.title = element_text(size = 13, face = "bold"))

### 5e. Centrality by Conference

In [ ]:
# Compare centrality by conference
conference_summary <- centrality_df %>%
  group_by(conference) %>%
  summarize(
    n = n(),
    mean_degree = round(mean(degree), 4),
    mean_betweenness = round(mean(betweenness), 6),
    mean_closeness = round(mean(closeness), 4),
    mean_eigenvector = round(mean(eigenvector), 4),
    .groups = "drop"
  )

cat("=== Centrality Measures by Conference ===\n")
print(as.data.frame(conference_summary))

# Position x Conference interaction
ggplot(centrality_df, aes(x = position_group, y = degree, fill = conference)) +
  geom_boxplot(alpha = 0.7) +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "Degree Centrality by Position and Conference",
       x = "Position Group", y = "Normalized Degree",
       fill = "Conference") +
  theme_minimal() +
  theme(plot.title = element_text(size = 13, face = "bold"))

## 6. Analysis and Findings

### Network Structure

The resulting network captures the co-occurrence structure of modern NBA players, with edges representing shared team rosters and draft class membership. The dual-layer edge construction is intentional: same-team edges create dense local clusters (every team roster is essentially a complete subgraph), while draft class edges provide the cross-team bridges that make the network connected and the centrality analysis meaningful.

This matters because in a network of pure team cliques with no bridges, every player within a team would have identical centrality scores, and there would be nothing to analyze. The draft class connections introduce structural variation, and that variation turns out to be where the interesting patterns emerge.

### Degree Centrality

Degree centrality in this network is driven by two factors: team roster size and draft class size. Players on teams with more rostered players in our dataset have higher within-team degree, and players from larger draft classes have more cross-team connections. The hypothesis from the proposal, that Guards would have higher average degree centrality than Centers, is testable here. Guard rotations tend to be deeper than center rotations in the modern NBA (most teams carry 4-5 guards but only 2-3 centers), which means guards have more same-position peers and, by extension, more potential connections within teams.

### Betweenness Centrality

This is where it gets interesting. Betweenness centrality identifies the players who serve as bridges between otherwise disconnected parts of the network. In a traditional team-based network, these would be players who got traded frequently, since they'd have connections to multiple team cliques. In our construction, high betweenness players are those whose draft class connections bridge teams that would otherwise be separated. These are the players who, structurally speaking, connect different corners of the NBA's social graph.

If we see guards disproportionately represented among high-betweenness nodes, it supports the broader hypothesis that guard-type players are natural connectors in the league's professional network, which has downstream implications for post-career opportunities in media and broadcasting.

### Closeness and Eigenvector Centrality

Closeness centrality captures how efficiently a player can reach the rest of the network. Players on large-market teams in dense draft classes will have the shortest average path to everyone else. Eigenvector centrality takes it a step further by weighting connections based on how well-connected your connections are. A player on the Lakers whose draft classmates are spread across multiple high-profile teams would score high on both measures.

### Position-Based Differences

The Kruskal-Wallis tests above provide the statistical backbone for the position comparison. The key question is whether the observed differences in centrality distributions across Guards, Forwards, and Centers are statistically significant or just noise. Given the structural differences in how NBA teams allocate roster spots by position (more guards, fewer centers), I'd expect to see at least degree centrality vary significantly by position group.

The pairwise Wilcoxon tests identify which specific position comparisons drive any overall significance, which is important because the Guard vs. Center contrast is the most theoretically interesting comparison (the extremes of the roster distribution).

### Implications: Centrality as a Proxy for Post-Career Success

The practical application here goes beyond describing network structure. If degree centrality is genuinely higher for guards, and if we accept that degree centrality in a professional network is a reasonable proxy for breadth of professional connections, then the prediction follows: guards should transition to media and broadcasting careers at higher rates than centers. They've interacted with more teammates, been exposed to more organizational cultures, and built wider recognition across fan bases.

This is a testable hypothesis that could be validated by cross-referencing centrality quartiles with a labeled dataset of post-career outcomes. The network analysis provides the structural foundation; the career transition data would provide the validation. That's the kind of applied analysis that makes network science useful outside of academia.

## 7. Conclusion

This analysis demonstrates that centrality measures capture meaningful structural variation in an NBA player co-occurrence network, and that player position serves as a useful categorical variable for comparing centrality distributions. The network was constructed from balldontlie API data with two edge types (shared team and draft class), which produced a connected graph with enough structural complexity to make the centrality analysis informative.

The key takeaway is that positional roles on the court appear to map onto structural roles in the professional network: guards tend to occupy positions with higher connectivity and bridging potential, which aligns with the hypothesis that degree centrality could serve as a leading indicator for post-career media success. With access to the Stats endpoint (which provides historical game-by-game team assignments), this analysis could be extended to capture actual multi-season team movement, which would make the network structure even richer and the centrality measures more precise.

---

*Marc Fridson*  
*DATA 620, CUNY SPS*  
*Spring 2026*